# 08 · Diagnostics: ETH EMOS anomaly + permutation p-values (CPU only)

No GPU needed: **Runtime → Change runtime type → CPU**. Uses results already on Drive.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/kronos-agent'
CODE = f'{ROOT}/code'

# Unpack code from Drive. Set UPDATE = True once after uploading a new kronos-agent.zip.
import os, shutil, zipfile, sys, glob
UPDATE = False
ZIP = f'{ROOT}/kronos-agent.zip'
if UPDATE or not os.path.exists(f'{CODE}/scripts/run_permutation.py'):
    assert os.path.exists(ZIP), f'upload kronos-agent.zip to {ROOT}/ first'
    shutil.rmtree(CODE, ignore_errors=True)
    zipfile.ZipFile(ZIP).extractall(CODE)

%pip install -q ccxt yfinance
%pip install -q -e "$CODE" --no-deps
if CODE not in sys.path:
    sys.path.insert(0, CODE)

EQ1 = ' '.join(f'configs/confirm/{t}.yaml' for t in ['qqq', 'iwm', 'aapl', 'msft', 'nvda', 'amzn', 'tsla', 'jpm'])
ALL = 'configs/artifact/spy.yaml ' + EQ1
C2 = ' '.join(sorted(os.path.relpath(p, CODE) for p in glob.glob(f'{CODE}/configs/confirm2/*.yaml')))
print(len(ALL.split()), len(C2.split()))

## Step 1 · Why EMOS hurts Chronos on ETH (~1 min)

In [ ]:
!cd $CODE && python scripts/diagnose_emos.py --root $ROOT --config configs/kronos_b_crypto/eth.yaml --run chronos64 "lookback=64;model_family=chronos;model_id=amazon/chronos-t5-small;amp=false" --run kronos64 "lookback=64"

## Step 2 · Permutation p-values next to the HAC ones (~5–10 min)

In [ ]:
!cd $CODE && python scripts/run_permutation.py --root $ROOT --name artifact --mode volbins fixbins labels --config $ALL --partial
!cd $CODE && python scripts/run_permutation.py --root $ROOT --name prereg2 --mode labels --config $C2
!cd $CODE && python scripts/run_permutation.py --root $ROOT --name prereg1_equities --mode volbins --config $EQ1

## Step 3 · Robustness: Phase B with the pooled EMOS spread estimator (~5 min)

In [ ]:
EQB = ' '.join(sorted(os.path.relpath(p, CODE) for p in glob.glob(f'{CODE}/configs/kronos_b/*.yaml')))
CHR = 'model_family=chronos;model_id=amazon/chronos-t5-small;amp=false'
!cd $CODE && python scripts/run_phaseb.py --root $ROOT --config $EQB --k-mode pooled --out phaseb_pooled --run kronos64 "lookback=64" --run kronos256 "lookback=256" --run chronos64 "lookback=64;$CHR" --run chronos256 "lookback=256;$CHR"
!cd $CODE && python scripts/run_phaseb.py --root $ROOT --config configs/kronos_b_crypto/eth.yaml --k-mode pooled --out phaseb_pooled_eth --run kronos64 "lookback=64" --run chronos64 "lookback=64;$CHR"